In [3]:
"""
Figure 1: Comparison of selected participatory mapping tools.
"""

import os

import matplotlib as mpl
import matplotlib.font_manager as fm
import matplotlib.patches as patches
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# Style — user-supplied block (e-Ukraine if installed, else DejaVu Sans)
# ---------------------------------------------------------------------------
_eu_fonts = [f for f in fm.findSystemFonts()
             if "e-Ukraine" in f or "eUkraine" in f or "e_Ukraine" in f]
if _eu_fonts:
    for fp in _eu_fonts:
        fm.fontManager.addfont(fp)
    _eu_name = fm.FontProperties(fname=_eu_fonts[0]).get_name()
else:
    _eu_name = "DejaVu Sans"

plt.rcParams.update({
    "font.family":     _eu_name,
    "font.size":       8,
    "axes.titlesize":  8,
    "axes.labelsize":  8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "figure.dpi":      300,
    "pdf.fonttype":    42,   # editable text in vector PDF
    "ps.fonttype":     42,
})

OUTDIR = "C:/Users/karasoo1/Ukrainability"
W_IN   = 18 / 2.54   # 18 cm

# ---------------------------------------------------------------------------
# Colour palette
# ---------------------------------------------------------------------------
COL_YES              = "#D6E8F5"
COL_NO               = "#F8D7D7"
COL_NEUTRAL          = "#F2F2F2"
COL_HEADER           = "#3B5A7A"
COL_HEADER_HIGHLIGHT = "#1F4068"   # slightly darker for "(this study)"
COL_FEATURE_COL      = "#E8ECF1"
COL_BORDER           = "#9AA5B1"
COL_TEXT_DARK        = "#1A1A1A"
COL_TEXT_LIGHT       = "#FFFFFF"

# ---------------------------------------------------------------------------
# Headers — three trailing "app"s and one "e.g.," dropped (see module docstring)
# ---------------------------------------------------------------------------
columns = [
    "Feature",
    "Chatbot\n(this study)",
    "Maptionnaire",
    "ArcGIS\ncrowdsourcing\n(Survey 123)",
    "GIS Cloud\n(Mobile Data\nCollection)",
    "KoboToolbox\n(KoboCollect)",
    "Open Data Kit\n(ODK Collect)",
]

# ---------------------------------------------------------------------------
# Body rows
# Each row: (feature label, [(cell text, classification ∈ {"yes","no","neutral"}), ...])
# ---------------------------------------------------------------------------
rows = [
    ("Open-source", [
        ("Yes",  "yes"),
        ("No",   "no"),
        ("No",   "no"),
        ("No",   "no"),
        ("Yes",  "yes"),
        ("Yes",  "yes"),
    ]),
    ("Embedded in\nexisting app", [
        ("Yes (Telegram)",              "yes"),
        ("No (standalone\nweb app)",    "no"),
        ("No (standalone\nweb/mobile)", "no"),
        ("No (standalone\nweb/mobile)", "no"),
        ("No (standalone\nweb/mobile)", "no"),
        ("No (standalone\nmobile app)", "no"),
    ]),
    ("Does not\nrequire app\ninstallation", [
        ("Yes (Telegram\nusers)",      "yes"),
        ("Yes",                        "yes"),
        ("No",                         "no"),
        ("No (customised\napp)",       "no"),
        ("No\n(KoboCollect)",          "no"),
        ("No\n(ODK Collect)",          "no"),
    ]),
    ("Free-text,\nvoice, and\nvideo input", [
        ("Yes", "yes"),
        ("Yes", "yes"),
        ("Yes", "yes"),
        ("Yes", "yes"),
        ("Yes", "yes"),
        ("Yes", "yes"),
    ]),
    ("Works offline", [
        ("No",  "no"),
        ("No",  "no"),
        ("Yes", "yes"),
        ("Yes", "yes"),
        ("Yes", "yes"),
        ("Yes", "yes"),
    ]),
    ("Reciprocal\nvalue to\nparticipants", [
        ("Yes (mini app\nintegration)", "yes"),
        ("Yes (depends\non setup)",     "yes"),
        ("Yes (depends\non setup)",     "yes"),
        ("Yes (depends\non setup)",     "yes"),
        ("No",                          "no"),
        ("No",                          "no"),
    ]),
    ("Response\nreview and\nmodification", [
        ("Yes (user can\nmodify any\nresponse)", "yes"),
        ("Yes (via My\nresponses)",              "yes"),
        ("Yes\n(via Outbox)",                    "yes"),
        ("Yes (via Info\nPanel)",                "yes"),
        ("Yes (via Edit\nSaved Form)",           "yes"),
        ("Yes (editing\nDraft)",                 "yes"),
    ]),
    ("Programming\nis not\nrequired", [
        ("No\n(Telegram API)",   "no"),
        ("Yes (GUI)",            "yes"),
        ("Yes (GUI)",            "yes"),
        ("Yes (GUI)",            "yes"),
        ("Yes\n(XLSForm)", "yes"),
        ("Yes\n(XLSForm)", "yes"),
    ]),
    ("Cost", [
        ("Free\n(self-hosting)",                     "yes"),
        ("Commercial\n(project\nlicense)",           "no"),
        ("Commercial\n(ArcGIS\nlicense)",            "no"),
        ("Freemium\n(limited\nfree tier)",           "neutral"),
        ("Free for\nhumanitarian\nuse; paid\nplans", "neutral"),
        ("Free\n(self-hosting),\ncommercial",        "neutral"),
    ]),
]

col_weights  = [1.55, 1.20, 1.15, 1.30, 1.20, 1.25, 1.25]
total_weight = sum(col_weights)
col_fracs    = [w / total_weight for w in col_weights]

# ---------------------------------------------------------------------------
# Layout
# ---------------------------------------------------------------------------
fig_height_in = 7.5     # taller than v1 to absorb 3- and 4-line cells
fig, ax = plt.subplots(figsize=(W_IN, fig_height_in))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_aspect("auto")
ax.axis("off")

header_h_frac    = 0.115
body_total_frac  = 1.0 - header_h_frac
body_row_h_frac  = body_total_frac / len(rows)

x_edges = [0.0]
for f in col_fracs:
    x_edges.append(x_edges[-1] + f)

# ---------- Header row ----------
y_header_top    = 1.0
y_header_bottom = 1.0 - header_h_frac

for j, col_name in enumerate(columns):
    x0, x1 = x_edges[j], x_edges[j + 1]
    is_this_study = (j == 1)
    bg = COL_HEADER_HIGHLIGHT if is_this_study else COL_HEADER
    ax.add_patch(patches.Rectangle(
        (x0, y_header_bottom), x1 - x0, header_h_frac,
        facecolor=bg, edgecolor=COL_BORDER, linewidth=0.6,
    ))
    ax.text(
        (x0 + x1) / 2, (y_header_bottom + y_header_top) / 2,
        col_name,
        ha="center", va="center",
        fontsize=8, color=COL_TEXT_LIGHT, fontweight="bold",
    )

# ---------- Body rows ----------
def cell_colour(classification: str) -> str:
    return {"yes": COL_YES, "no": COL_NO, "neutral": COL_NEUTRAL}[classification]

for i, (feature, cells) in enumerate(rows):
    y_top = y_header_bottom - i * body_row_h_frac
    y_bot = y_top - body_row_h_frac
    y_mid = (y_top + y_bot) / 2

    # Feature-label cell — text centred horizontally to avoid the left-edge
    x0, x1 = x_edges[0], x_edges[1]
    ax.add_patch(patches.Rectangle(
        (x0, y_bot), x1 - x0, body_row_h_frac,
        facecolor=COL_FEATURE_COL, edgecolor=COL_BORDER, linewidth=0.5,
    ))
    ax.text(
        (x0 + x1) / 2, y_mid, feature,
        ha="center", va="center",
        fontsize=8, color=COL_TEXT_DARK, fontweight="bold",
    )

    # Tool cells
    for j, (txt, cls) in enumerate(cells, start=1):
        cx0, cx1 = x_edges[j], x_edges[j + 1]
        ax.add_patch(patches.Rectangle(
            (cx0, y_bot), cx1 - cx0, body_row_h_frac,
            facecolor=cell_colour(cls), edgecolor=COL_BORDER, linewidth=0.5,
        ))
        ax.text(
            (cx0 + cx1) / 2, y_mid, txt,
            ha="center", va="center",
            fontsize=7.5, color=COL_TEXT_DARK,
        )

# ---------- Legend ----------
legend_y     = -0.055
legend_box_w = 0.022
legend_box_h = 0.032

legend_items = [
    ("Yes",                 COL_YES),
    ("No",                  COL_NO),
    ("Qualitative / mixed", COL_NEUTRAL),
]

char_w_axes = 0.0095
total_w = sum(legend_box_w + 0.005 + len(lbl) * char_w_axes + 0.020
              for lbl, _ in legend_items) - 0.020
x_cursor = (1.0 - total_w) / 2

for label, colour in legend_items:
    ax.add_patch(patches.Rectangle(
        (x_cursor, legend_y), legend_box_w, legend_box_h,
        facecolor=colour, edgecolor=COL_BORDER, linewidth=0.5,
        transform=ax.transAxes, clip_on=False,
    ))
    ax.text(
        x_cursor + legend_box_w + 0.005, legend_y + legend_box_h / 2,
        label, ha="left", va="center",
        fontsize=8, color=COL_TEXT_DARK,
        transform=ax.transAxes, clip_on=False,
    )
    x_cursor += legend_box_w + 0.005 + len(label) * char_w_axes + 0.020

# ---------------------------------------------------------------------------
# Save
# ---------------------------------------------------------------------------
plt.subplots_adjust(left=0.005, right=0.995, top=0.99, bottom=0.06)

os.makedirs(OUTDIR, exist_ok=True)
out_pdf = os.path.join(OUTDIR, "Figure_1_tool_comparison.pdf")
out_png = os.path.join(OUTDIR, "Figure_1_tool_comparison.png")

fig.savefig(out_pdf, bbox_inches="tight", pad_inches=0.05)
fig.savefig(out_png, bbox_inches="tight", pad_inches=0.05, dpi=900)

plt.close(fig)
print(f"Saved: {out_pdf}")
print(f"Saved: {out_png}")

Saved: C:/Users/karasoo1/Ukrainability\Figure_1_tool_comparison.pdf
Saved: C:/Users/karasoo1/Ukrainability\Figure_1_tool_comparison.png


In [1]:
"""
Figure 5, Figures S2-S3 (descriptive statistics)

"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.font_manager as fm
import matplotlib.patheffects as pe
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings("ignore")

# ── Font ─────────────────────────────────────────────────────────────────────
_eu_fonts = [f for f in fm.findSystemFonts()
             if "e-Ukraine" in f or "eUkraine" in f or "e_Ukraine" in f]
if _eu_fonts:
    for fp in _eu_fonts:
        fm.fontManager.addfont(fp)
    _eu_name = fm.FontProperties(fname=_eu_fonts[0]).get_name()
else:
    _eu_name = "DejaVu Sans"

plt.rcParams.update({
    "font.family":     _eu_name,
    "font.size":       8,
    "axes.titlesize":  8,
    "axes.labelsize":  8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "figure.dpi":      300,
})

OUTDIR = "C:/Users/karasoo1/Ukrainability"
W_IN   = 18 / 2.54
PNTD   = "Prefer not to disclose"

# ── Data ──────────────────────────────────────────────────────────────────────
df = pd.read_excel(f"{OUTDIR}/english_data_for_sharing.xlsx", sheet_name="english_data")
df["duration_visit"] = df["duration_visit"].replace("1–2 hours", "1-2 hours")

emap = {
    "Not enjoyable at all": "Not enjoyable",
    "Not enjoyable":        "Not enjoyable",
    "Neutral":              "Not enjoyable",
    "Enjoyable":            "Enjoyable",
    "Very enjoyable":       "Enjoyable",
}
df["enjoyment_bin"] = df["enjoyment"].map(emap)
target_order = ["Not enjoyable", "Enjoyable"]

CB   = {"Not enjoyable": "#D55E00", "Enjoyable": "#009E73"}
FREQ = "#56B4E9"

# Text outline for better contrast
outline_effect = [pe.withStroke(linewidth=1.2, foreground=(0, 0, 0, 0.4))]

# ── Utilities ─────────────────────────────────────────────────────────────────

def _save(fig, stem):
    fig.savefig(f"{OUTDIR}/{stem}.png", bbox_inches="tight", dpi=300, format="png")
    fig.savefig(f"{OUTDIR}/{stem}.pdf", bbox_inches="tight",           format="pdf")
    plt.close(fig)
    print(f"  ✓ {stem}.png / .pdf")


def _pntd_last(lst):
    """Move PNTD to end of list."""
    return [v for v in lst if v != PNTD] + ([PNTD] if PNTD in lst else [])


def _sort_profile(pct_df):
    """Sort rows by Enjoyable % descending; PNTD always last."""
    main = pct_df.drop(PNTD, errors="ignore")
    ordered = main["Enjoyable"].sort_values(ascending=False).index.tolist()
    if PNTD in pct_df.index:
        ordered.append(PNTD)
    return ordered


def _count_bars_single(series):
    return int(series.dropna().nunique())


def _count_bars_multi(series):
    vals = set()
    for v in series.dropna():
        for item in str(v).split(";"):
            s = item.strip()
            if s:
                vals.add(s)
    return len(vals)


def _row_h(n_bars):
    """Height in inches for one subplot row with n_bars."""
    return max(1.3, n_bars * 0.32)


def _build_fig(nrows, ncols, bars_per_row, legend=False):
    """
    Create figure + axes.
    bars_per_row: list of bar counts, one per row (max across columns).
    """
    row_heights = [_row_h(n) for n in bars_per_row]
    legend_pad  = 0.35 if legend else 0.0
    overhead    = nrows * 0.65
    fig_h       = sum(row_heights) + overhead + legend_pad

    gs_kw = {"height_ratios": row_heights} if nrows > 1 else {}
    fig, axs = plt.subplots(nrows, ncols, figsize=(W_IN, fig_h),
                            gridspec_kw=gs_kw)

    # Normalise to 2-D list
    if nrows == 1 and ncols == 1:
        axs = [[axs]]
    elif nrows == 1:
        axs = [list(axs)]
    elif ncols == 1:
        axs = [[a] for a in axs]
    else:
        axs = [list(row) for row in axs]
    return fig, axs, legend_pad, fig_h


def _legend(fig, legend_pad, fig_h):
    handles = [Patch(facecolor=CB[c], edgecolor="white", label=c)
               for c in target_order]
    # fraction of figure height that the legend strip occupies
    frac = legend_pad / fig_h
    fig.legend(handles=handles, loc="lower center", ncol=2,
               fontsize=8, frameon=False,
               bbox_to_anchor=(0.5, 0.02),
               bbox_transform=fig.transFigure)
    return frac


# ═══════════════════════════════════════════════════════════════════════════
# INDIVIDUAL AXES DRAWING
# ═══════════════════════════════════════════════════════════════════════════

def freq_single(ax, series, label, order=None):
    vc = series.dropna().value_counts()
    n  = series.dropna().shape[0]
    pct  = (vc / n) * 100
    
    if order:
        idx = [o for o in order if o in pct.index]
        if PNTD in pct.index and PNTD not in idx:
            idx.append(PNTD)
    else:
        idx = _pntd_last(pct.sort_values(ascending=False).index.tolist())
        
    data = pct.reindex(idx).iloc[::-1]   # reverse so top item is at top

    bars = ax.barh(range(len(data)), data.values,
                   color=FREQ, edgecolor="white", height=0.6)
    ax.set_yticks(range(len(data)))
    ax.set_yticklabels(data.index)
    ax.set_xlabel("% of respondents")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
    ax.set_xlim(0, data.max() * 1.32)
    ax.spines[["top", "right"]].set_visible(False)
    for bar, v in zip(bars, data.values):
        ax.text(bar.get_width() + 0.4,
                bar.get_y() + bar.get_height() / 2,
                f"{v:.1f}%", va="center", fontsize=7)
    if label:
        ax.set_title(f"({label})", loc="left", fontsize=8)


def freq_multi(ax, series, label):
    counts, n = {}, series.dropna().shape[0]
    for v in series.dropna():
        for item in str(v).split(";"):
            s = item.strip()
            if s:
                counts[s] = counts.get(s, 0) + 1
    pct  = (pd.Series(counts) / n) * 100
    idx  = _pntd_last(pct.sort_values(ascending=False).index.tolist())
    data = pct.reindex(idx).iloc[::-1]

    bars = ax.barh(range(len(data)), data.values,
                   color=FREQ, edgecolor="white", height=0.6)
    ax.set_yticks(range(len(data)))
    ax.set_yticklabels(data.index)
    ax.set_xlabel("% of respondents")
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
    ax.set_xlim(0, data.max() * 1.32)
    ax.spines[["top", "right"]].set_visible(False)
    for bar, v in zip(bars, data.values):
        ax.text(bar.get_width() + 0.4,
                bar.get_y() + bar.get_height() / 2,
                f"{v:.1f}%", va="center", fontsize=7)
    if label:
        ax.set_title(f"({label})", loc="left", fontsize=8)


def profile_single(ax, col, label, order=None, min_n=5):
    ct  = pd.crosstab(df[col], df["enjoyment_bin"])
    ct  = ct.reindex(columns=target_order).fillna(0)
    ct  = ct[ct.sum(axis=1) >= min_n]
    pct = ct.div(ct.sum(axis=1), axis=0) * 100
    
    if order:
        idx = [o for o in order if o in pct.index]
        if PNTD in pct.index and PNTD not in idx:
            idx.append(PNTD)
    else:
        idx = _sort_profile(pct)
        
    ct  = ct.reindex(idx);  pct = pct.reindex(idx)
    # Reverse for barh (top item ends up at top)
    ct_r = ct.iloc[::-1];  pct_r = pct.iloc[::-1]

    left = np.zeros(len(pct_r))
    for cat in target_order:
        vals = pct_r[cat].values if cat in pct_r.columns else np.zeros(len(pct_r))
        ax.barh(range(len(pct_r)), vals, left=left,
                color=CB[cat], edgecolor="white", height=0.6)
        for j, (v, l) in enumerate(zip(vals, left)):
            if v > 9:
                ax.text(l + v / 2, j, f"{v:.0f}%",
                        ha="center", va="center", fontsize=7, color="white",
                        path_effects=outline_effect)
        left += vals

    ax.set_yticks(range(len(pct_r)))
    ax.set_yticklabels(pct_r.index)
    for j, row_idx in enumerate(pct_r.index):
        ax.text(101.5, j, f"n={int(ct_r.loc[row_idx].sum())}",
                va="center", fontsize=8, color="grey")
    ax.set_xlim(0, 116)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
    ax.set_xlabel("% of respondents")
    ax.spines[["top", "right"]].set_visible(False)
    if label:
        ax.set_title(f"({label})", loc="left", fontsize=8)


def profile_multi(ax, col, label, min_n=5):
    opts = set()
    for v in df[col].dropna():
        for item in str(v).split(";"):
            s = item.strip()
            if s:
                opts.add(s)
    rows = {}
    for opt in opts:
        mask = df[col].dropna().apply(lambda x: opt in str(x).split(";"))
        sub  = df.loc[mask.index[mask], "enjoyment_bin"]
        rows[opt] = sub.value_counts().reindex(target_order).fillna(0)
    ct  = pd.DataFrame(rows).T
    ct  = ct[ct.sum(axis=1) >= min_n]
    pct = ct.div(ct.sum(axis=1), axis=0) * 100
    idx = _sort_profile(pct)
    ct  = ct.reindex(idx);  pct = pct.reindex(idx)
    ct_r = ct.iloc[::-1];   pct_r = pct.iloc[::-1]

    left = np.zeros(len(pct_r))
    for cat in target_order:
        vals = pct_r[cat].values
        ax.barh(range(len(pct_r)), vals, left=left,
                color=CB[cat], edgecolor="white", height=0.6)
        for j, (v, l) in enumerate(zip(vals, left)):
            if v > 9:
                ax.text(l + v / 2, j, f"{v:.0f}%",
                        ha="center", va="center", fontsize=7, color="white",
                        path_effects=outline_effect)
        left += vals

    ax.set_yticks(range(len(pct_r)))
    ax.set_yticklabels(pct_r.index)
    for j, row_idx in enumerate(pct_r.index):
        ax.text(101.5, j, f"n={int(ct_r.loc[row_idx].sum())}",
                va="center", fontsize=8, color="grey")
    ax.set_xlim(0, 116)
    ax.xaxis.set_major_formatter(mticker.PercentFormatter(decimals=0))
    ax.set_xlabel("% of respondents")
    ax.spines[["top", "right"]].set_visible(False)
    if label:
        ax.set_title(f"({label})", loc="left", fontsize=8)


# ═══════════════════════════════════════════════════════════════════════════
# FIGURES
# ═══════════════════════════════════════════════════════════════════════════

print("Saving charts...")

# Income custom ordering high to low
inc_order = ["20000+ UAH", "10001-20000 UAH", "5001-10000 UAH", "0-5000 UAH"]

# ── 1a. Socioeconomic — Frequency ─────────────────────────────────────────────
r0 = max(_count_bars_single(df["age"]),    _count_bars_single(df["gender"]))
r1 = max(_count_bars_single(df["income"]), _count_bars_single(df["occupation"]))
fig, axs, _, _ = _build_fig(2, 2, [r0, r1])
freq_single(axs[0][0], df["age"],        "a")
freq_single(axs[0][1], df["gender"],     "b")
freq_single(axs[1][0], df["income"],     "c", order=inc_order)
freq_single(axs[1][1], df["occupation"], "d")
fig.tight_layout(pad=1.5, w_pad=2.5, rect=[0, 0, 0.96, 1])
_save(fig, "freq_socioeconomic")

# ── 1b. Socioeconomic — Profile ───────────────────────────────────────────────
fig, axs, lpad, fh = _build_fig(2, 2, [r0, r1], legend=True)
profile_single(axs[0][0], "age",        "a")
profile_single(axs[0][1], "gender",     "b")
profile_single(axs[1][0], "income",     "c", order=inc_order)
profile_single(axs[1][1], "occupation", "d")
lfrac = _legend(fig, lpad, fh)
fig.tight_layout(pad=1.5, w_pad=2.5, rect=[0, lfrac, 0.96, 1])
_save(fig, "profile_socioeconomic")

# ── 2a. Visit characteristics — Frequency ─────────────────────────────────────
r0 = max(_count_bars_single(df["regularity"]), _count_bars_single(df["duration_visit"]))
r1 = _count_bars_single(df["noticed_changes"])
fig, axs, _, _ = _build_fig(2, 2, [r0, r1])
freq_single(axs[0][0], df["regularity"],      "a")
freq_single(axs[0][1], df["duration_visit"],  "b")
freq_single(axs[1][0], df["noticed_changes"], "c")
axs[1][1].axis("off")
fig.tight_layout(pad=1.5, w_pad=2.5, rect=[0, 0, 0.96, 1])
_save(fig, "freq_visit_characteristics")

# ── 2b. Visit characteristics — Profile ──────────────────────────────────────
fig, axs, lpad, fh = _build_fig(2, 2, [r0, r1], legend=True)
profile_single(axs[0][0], "regularity",      "a")
profile_single(axs[0][1], "duration_visit",  "b")
profile_single(axs[1][0], "noticed_changes", "c")
axs[1][1].axis("off")
lfrac = _legend(fig, lpad, fh)
fig.tight_layout(pad=1.5, w_pad=2.5, rect=[0, lfrac, 0.96, 1])
_save(fig, "profile_visit_characteristics")

# ── 3a. Multi-choice — Frequency ──────────────────────────────────────────────
r0 = max(_count_bars_multi(df["purpose_visit"]), _count_bars_multi(df["wishlist"]))
r1 = max(_count_bars_multi(df["visitor_type"]),  _count_bars_multi(df["accessibility"]))
fig, axs, _, _ = _build_fig(2, 2, [r0, r1])
freq_multi(axs[0][0], df["purpose_visit"], "a")
freq_multi(axs[0][1], df["wishlist"],      "b")
freq_multi(axs[1][0], df["visitor_type"],  "c")
freq_multi(axs[1][1], df["accessibility"], "d")
fig.tight_layout(pad=1.5, w_pad=2.5, rect=[0, 0, 0.96, 1])
_save(fig, "freq_multichoice")

# ── 3b. Multi-choice — Profile ────────────────────────────────────────────────
fig, axs, lpad, fh = _build_fig(2, 2, [r0, r1], legend=True)
profile_multi(axs[0][0], "purpose_visit", "a")
profile_multi(axs[0][1], "wishlist",      "b")
profile_multi(axs[1][0], "visitor_type",  "c")
profile_multi(axs[1][1], "accessibility", "d")
lfrac = _legend(fig, lpad, fh)
fig.tight_layout(pad=1.5, w_pad=2.5, rect=[0, lfrac, 0.96, 1])
_save(fig, "profile_multichoice")

# ── 4. Enjoyment — Frequency (standalone) ─────────────────────────────────────
enj_order = ["Not enjoyable at all", "Not enjoyable", "Neutral",
             "Enjoyable", "Very enjoyable"]
fig, ax = plt.subplots(figsize=(W_IN, _row_h(len(enj_order)) + 0.8))
freq_single(ax, df["enjoyment"], "", order=enj_order)
fig.tight_layout(pad=1.5, w_pad=2.5, rect=[0, 0, 0.96, 1])
_save(fig, "freq_enjoyment")

print("\n✓ Done.")

Saving charts...
  ✓ freq_socioeconomic.png / .pdf
  ✓ profile_socioeconomic.png / .pdf
  ✓ freq_visit_characteristics.png / .pdf
  ✓ profile_visit_characteristics.png / .pdf
  ✓ freq_multichoice.png / .pdf
  ✓ profile_multichoice.png / .pdf
  ✓ freq_enjoyment.png / .pdf

✓ Done.


In [2]:
"""
Predictive models (Table 1, Figure 6, Table S2, Figure S4, Table S3)
Binary classification: 'Enjoyable' vs 'Not enjoyable'

Two classifiers compared:
  1. Random Forest  (primary model)
  2. CatBoost       (robustness check)
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_recall_fscore_support,
)

import shap
import warnings

warnings.filterwarnings("ignore")

# ═══════════════════════════════════════════════════════════════════════════
# 0. Style + e-Ukraine Font
# ═══════════════════════════════════════════════════════════════════════════
_eu_fonts = [
    f for f in fm.findSystemFonts()
    if "e-Ukraine" in f or "eUkraine" in f or "e_Ukraine" in f
]
if _eu_fonts:
    for fp in _eu_fonts:
        fm.fontManager.addfont(fp)
    _eu_name = fm.FontProperties(fname=_eu_fonts[0]).get_name()
    print(f"✓ Found e-Ukraine font: '{_eu_name}'")
else:
    _eu_name = "e-Ukraine"
    print(f"⚠ e-Ukraine not auto-detected; using '{_eu_name}' directly.")

plt.rcParams.update({
    "font.family": _eu_name,
    "font.size": 16,
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "figure.dpi": 150,
})

OUTDIR      = "C:/Users/karasoo1/Ukrainability"
CM_PER_INCH = 2.54
PANEL_W_IN  = 18 / CM_PER_INCH  # ~7.09 in
DPI_PNG     = 900

TARGET_ORDER = ["Not enjoyable", "Enjoyable"]
PANEL_LABELS = ["(a)", "(b)"]
TOP_N_BAR    = 10
TOP_N_DEP    = 6


def savefig_both(fig, path_no_ext, dpi=DPI_PNG):
    """Save figure as .png (900 dpi) and .pdf (vector)."""
    fig.savefig(f"{path_no_ext}.png", bbox_inches="tight", dpi=dpi, format="png")
    fig.savefig(f"{path_no_ext}.pdf", bbox_inches="tight", format="pdf")


# ═══════════════════════════════════════════════════════════════════════════
# 1. Load & Prepare
# ═══════════════════════════════════════════════════════════════════════════
df = pd.read_excel(f"{OUTDIR}/english_data_for_sharing.xlsx", sheet_name="english_data")

# Binary target: Negative + Neutral → "Not enjoyable"
enjoyment_map = {
    "Not enjoyable at all": "Not enjoyable",
    "Not enjoyable":        "Not enjoyable",
    "Neutral":              "Not enjoyable",
    "Enjoyable":            "Enjoyable",
    "Very enjoyable":       "Enjoyable",
}
df["enjoyment_bin"] = df["enjoyment"].map(enjoyment_map)
df["target"] = df["enjoyment_bin"].map(
    {v: i for i, v in enumerate(TARGET_ORDER)}
)

print("Target distribution (binary):")
print(df["enjoyment_bin"].value_counts().reindex(TARGET_ORDER))
print()

# ── 1b. Multi-choice → binary columns ──────────────────────────────────────
def explode_semicolon(series, prefix):
    all_vals = set()
    for v in series.dropna():
        for item in str(v).split(";"):
            item = item.strip()
            if item:
                all_vals.add(item)
    all_vals = sorted(all_vals)
    out = pd.DataFrame(
        0, index=series.index,
        columns=[f"{prefix}__{v}" for v in all_vals],
    )
    for i, v in series.items():
        if pd.isna(v):
            continue
        for item in str(v).split(";"):
            item = item.strip()
            if item:
                out.loc[i, f"{prefix}__{item}"] = 1
    return out

purpose_bin = explode_semicolon(df["purpose_visit"], "purpose")
access_bin  = explode_semicolon(df["accessibility"], "access")
visitor_bin = explode_semicolon(df["visitor_type"],  "visitor")

# ── 1c. Encode ordinal / nominal columns ───────────────────────────────────
age_order = ["18-25", "26-40", "41-60", "60+"]
df["age_ord"] = df["age"].map({v: i for i, v in enumerate(age_order)})

income_order = ["0-5000 UAH", "5001-10000 UAH", "10001-20000 UAH", "20000+ UAH"]
df["income_ord"] = df["income"].map({v: i for i, v in enumerate(income_order)})

df["is_female"] = (df["gender"] == "Female").astype(int)
df["is_male"]   = (df["gender"] == "Male").astype(int)

for occ in ["Working", "Student", "Retired", "Not working"]:
    df[f"occ_{occ.lower().replace(' ', '_')}"] = (
        df["occupation"] == occ
    ).astype(int)

reg_order = [
    "One-time visit", "Before war only", "Occasionally",
    "Regularly", "Frequently", "Very frequently",
]
df["regularity_ord"] = df["regularity"].map(
    {v: i for i, v in enumerate(reg_order)}
)

df["duration_ord"] = df["duration_visit"].replace("1–2 hours", "1-2 hours")
dur_order_clean = [
    "Passing by", "1-2 hours", "Most of the day", "Longer trip", "Few days",
]
df["duration_ord"] = df["duration_ord"].map(
    {v: i for i, v in enumerate(dur_order_clean)}
)

krem_order = [
    "Not local", "<1 year", "1-3 years", "3-10 years", "10-20 years", "Whole life",
]
df["residency_ord"] = df["kremenchuk"].map(
    {v: i for i, v in enumerate(krem_order)}
)

changes_map = {"Negative changes": 0, "No changes": 1, "Positive changes": 2}
df["changes_ord"] = df["noticed_changes"].map(changes_map)

df["n_activities"] = purpose_bin.sum(axis=1)

# ── 1d. Assemble feature matrix ────────────────────────────────────────────
socio_cols = [
    "age_ord", "income_ord", "is_female", "is_male",
    "occ_working", "occ_student", "occ_retired", "occ_not_working",
]
context_cols = [
    "regularity_ord", "duration_ord", "residency_ord",
    "changes_ord", "n_activities",
]

X = pd.concat(
    [purpose_bin, access_bin, visitor_bin, df[socio_cols], df[context_cols]],
    axis=1,
).astype(float)

y    = df["target"].values
mask = ~np.isnan(y)
X    = X.loc[mask].reset_index(drop=True)
y    = y[mask].astype(int)

# Fill ordinal NaNs (from "Prefer not to disclose") with column median
X = X.apply(lambda col: col.fillna(col.median()))

rename = {
    c: c.replace("purpose__", "Act: ")
        .replace("access__", "Acc: ")
        .replace("visitor__", "Vis: ")
    for c in X.columns
}
rename.update({
    "age_ord": "Age (ordinal)",
    "income_ord": "Income (ordinal)",
    "is_female": "Female",
    "is_male": "Male",
    "occ_working": "Employed",
    "occ_student": "Student",
    "occ_retired": "Retired",
    "occ_not_working": "Not employed",
    "regularity_ord": "Visit regularity (ordinal)",
    "duration_ord": "Visit duration (ordinal)",
    "residency_ord": "Residency length (ordinal)",
    "changes_ord": "Noticed changes (ordinal)",
    "n_activities": "Nr. activities selected",
})
X.columns = [rename.get(c, c) for c in X.columns]

print(f"Feature matrix: {X.shape[0]} obs × {X.shape[1]} features")
print(f"Target classes:  {np.bincount(y)}  (Not enjoyable / Enjoyable)\n")

# ═══════════════════════════════════════════════════════════════════════════
# 2. Model Definitions
# ═══════════════════════════════════════════════════════════════════════════
MODELS = {
    "rf": (
        "Random Forest",
        RandomForestClassifier(
            n_estimators=500,
            max_depth=5,
            min_samples_leaf=5,
            class_weight="balanced",
            random_state=42,
            n_jobs=-1,
        ),
    ),
    "catboost": (
        "CatBoost",
        CatBoostClassifier(
            iterations=300,
            depth=4,
            learning_rate=0.05,
            auto_class_weights="Balanced",
            random_seed=42,
            verbose=0,
        ),
    ),
}


# ═══════════════════════════════════════════════════════════════════════════
# 3. Cross-Validation Loop
# ═══════════════════════════════════════════════════════════════════════════
N_SPLITS = 5
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

cv_sheets   = {}
oof_preds   = {}
oof_reports = {}

for mkey, (mname, estimator) in MODELS.items():
    print(f"─── CV: {mname} ───")
    y_pred_oof = np.zeros_like(y)
    fold_rows  = []

    for fold, (tr_idx, te_idx) in enumerate(cv.split(X, y), 1):
        X_tr, X_te = X.iloc[tr_idx], X.iloc[te_idx]
        y_tr, y_te = y[tr_idx], y[te_idx]

        estimator.fit(X_tr, y_tr)
        preds = estimator.predict(X_te)
        y_pred_oof[te_idx] = preds

        acc = accuracy_score(y_te, preds)
        prec, rec, f1, _ = precision_recall_fscore_support(
            y_te, preds, labels=[0, 1], zero_division=0,
        )
        fold_rows.append({
            "Fold": f"Fold {fold}",
            "Accuracy": acc,
            "Precision (Not enjoyable)": prec[0],
            "Recall (Not enjoyable)":    rec[0],
            "F1 (Not enjoyable)":        f1[0],
            "Precision (Enjoyable)":     prec[1],
            "Recall (Enjoyable)":        rec[1],
            "F1 (Enjoyable)":            f1[1],
            "Macro F1": (f1[0] + f1[1]) / 2.0,
        })

    df_folds = pd.DataFrame(fold_rows)

    mean_vals = df_folds.select_dtypes(include="number").mean().to_dict()
    mean_vals["Fold"] = "Mean"
    sd_vals = df_folds.select_dtypes(include="number").std().to_dict()
    sd_vals["Fold"] = "SD"
    df_folds = pd.concat(
        [df_folds, pd.DataFrame([mean_vals, sd_vals])], ignore_index=True,
    )

    split_pct = f"{int((N_SPLITS-1)/N_SPLITS*100)}% Train / {int(1/N_SPLITS*100)}% Test"
    df_folds.insert(1, "Split Ratio", split_pct)

    cv_sheets[mkey] = df_folds
    oof_preds[mkey]  = y_pred_oof
    oof_reports[mkey] = classification_report(
        y, y_pred_oof, target_names=TARGET_ORDER, zero_division=0,
    )
    print(oof_reports[mkey])

# ── Summary sheet ───────────────────────────────────────────────────────────
summary_rows = []
for mkey, (mname, _) in MODELS.items():
    row = cv_sheets[mkey].loc[
        cv_sheets[mkey]["Fold"] == "Mean"
    ].iloc[0].to_dict()
    row["Model"] = mname
    row.pop("Fold", None)
    row.pop("Split Ratio", None)
    summary_rows.append(row)
df_summary = pd.DataFrame(summary_rows)
cols_order = ["Model"] + [c for c in df_summary.columns if c != "Model"]
df_summary = df_summary[cols_order]


# ═══════════════════════════════════════════════════════════════════════════
# 4. Refit on Full Data + SHAP
# ═══════════════════════════════════════════════════════════════════════════
shap_dict    = {}
feat_imp_dfs = {}

for mkey, (mname, estimator) in MODELS.items():
    print(f"─── SHAP: {mname} ───")

    estimator.fit(X, y)

    # ── Compute SHAP values ─────────────────────────────────────────────
    if mkey == "catboost":
        explainer = shap.TreeExplainer(estimator)
        sv_raw    = explainer.shap_values(X)          # 2D, positive class
        sv_list   = [-sv_raw, sv_raw]
    else:
        # RF → may return list-of-2 or 3D array
        explainer = shap.TreeExplainer(estimator)
        sv_raw    = explainer.shap_values(X)
        if isinstance(sv_raw, list):
            sv_list = sv_raw
        elif isinstance(sv_raw, np.ndarray) and sv_raw.ndim == 3:
            sv_list = [sv_raw[:, :, c] for c in range(sv_raw.shape[2])]
        else:
            sv_list = [-sv_raw, sv_raw]

    shap_dict[mkey] = sv_list

    # ── Feature importance table (avg |SHAP| across both classes) ───────
    mean_abs = np.mean(
        [np.abs(sv_list[c]).mean(axis=0) for c in range(2)], axis=0,
    )
    imp = (
        pd.Series(mean_abs, index=X.columns)
        .sort_values(ascending=False)
        .reset_index()
    )
    imp.columns = ["Feature", "Mean |SHAP| (avg 2 classes)"]
    feat_imp_dfs[mkey] = imp

print()

# ═══════════════════════════════════════════════════════════════════════════
# 5. Export all metrics
# ═══════════════════════════════════════════════════════════════════════════
xlsx_path = f"{OUTDIR}/model_comparison_report.xlsx"
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    df_summary.to_excel(writer, sheet_name="CV_Summary", index=False)
    for mkey, (mname, _) in MODELS.items():
        cv_sheets[mkey].to_excel(
            writer, sheet_name=f"CV_{mkey}", index=False,
        )
    for mkey in MODELS:
        feat_imp_dfs[mkey].to_excel(
            writer, sheet_name=f"SHAP_{mkey}", index=False,
        )
print(f"✓ Exported: {xlsx_path}")

# ═══════════════════════════════════════════════════════════════════════════
# 6. Plots
# ═══════════════════════════════════════════════════════════════════════════

# ── 6a. Confusion Matrices ──────────────────────────────────────────────────
for mkey, (mname, _) in MODELS.items():
    cm = confusion_matrix(y, oof_preds[mkey])
    fig, ax = plt.subplots(figsize=(5, 4.5))
    sns.heatmap(
        cm, annot=True, fmt="d", cmap="Blues",
        annot_kws={"size": 14},
        xticklabels=TARGET_ORDER, yticklabels=TARGET_ORDER, ax=ax,
    )
    ax.set_xlabel("Predicted", fontsize=14)
    ax.set_ylabel("Actual", fontsize=14)
    ax.tick_params(labelsize=12)
    fig.tight_layout()
    savefig_both(fig, f"{OUTDIR}/confusion_matrix_{mkey}")
    plt.close()
    print(f"✓ confusion_matrix_{mkey}.png / .pdf")

# ── 6b. SHAP Beeswarm Summaries (two-panel per model) ───────────────────────
for mkey, (mname, _) in MODELS.items():
    sv_list = shap_dict[mkey]
    fig, axes = plt.subplots(
        1, 2, figsize=(2 * PANEL_W_IN, PANEL_W_IN * 1.1),
    )
    for cls_idx in range(2):
        ax = axes[cls_idx]
        plt.sca(ax)
        shap.summary_plot(
            sv_list[cls_idx], X,
            max_display=25, show=False, plot_size=None,
        )
        ax.set_title(PANEL_LABELS[cls_idx], fontsize=18, pad=10)
        ax.set_xlabel("SHAP value", fontsize=18)
        ax.tick_params(axis="both", labelsize=16)

    for cb_ax in fig.axes:
        if cb_ax not in axes:
            cb_ax.set_ylabel("Feature value", fontsize=16)
            cb_ax.tick_params(labelsize=14)

    fig.tight_layout(pad=2.0)
    savefig_both(fig, f"{OUTDIR}/shap_beeswarm_{mkey}")
    plt.close()
    print(f"✓ shap_beeswarm_{mkey}.png / .pdf")

# ── 6c. SHAP Importance Bar Charts (two-panel per model) ────────────────────
for mkey, (mname, _) in MODELS.items():
    sv_list = shap_dict[mkey]
    fig, axes = plt.subplots(
        1, 2, figsize=(2 * PANEL_W_IN, PANEL_W_IN * 0.75),
    )
    for cls_idx, cls_name in enumerate(TARGET_ORDER):
        ax       = axes[cls_idx]
        sv       = sv_list[cls_idx]
        mean_abs = np.abs(sv).mean(axis=0)
        feat_imp = (
            pd.Series(mean_abs, index=X.columns)
            .sort_values(ascending=False)
        )
        feat_imp.head(TOP_N_BAR).iloc[::-1].plot.barh(
            ax=ax, color="#4a90d9", edgecolor="white",
        )
        ax.set_xlabel("Mean |SHAP| value", fontsize=18)
        ax.set_title(PANEL_LABELS[cls_idx], fontsize=18, pad=10)
        ax.tick_params(axis="both", labelsize=16)
        ax.spines[["top", "right"]].set_visible(False)

    fig.tight_layout(pad=2.0)
    savefig_both(fig, f"{OUTDIR}/shap_importance_{mkey}")
    plt.close()
    print(f"✓ shap_importance_{mkey}.png / .pdf")

# ── 6d. SHAP Dependence Plots (two-panel per model) ─────────────────────────
for mkey, (mname, _) in MODELS.items():
    sv_list = shap_dict[mkey]
    mean_abs_both = np.mean(
        [np.abs(sv_list[c]).mean(axis=0) for c in range(2)], axis=0,
    )
    top_feats = (
        pd.Series(mean_abs_both, index=X.columns)
        .sort_values(ascending=False)
        .head(TOP_N_DEP)
        .index.tolist()
    )

    n_feat = len(top_feats)
    n_rows = int(np.ceil(n_feat / 3))
    fig, axes = plt.subplots(
        n_rows, 6,
        figsize=(2 * PANEL_W_IN, PANEL_W_IN * 0.45 * n_rows),
    )
    if axes.ndim == 1:
        axes = axes.reshape(1, -1)

    for cls_idx in range(2):
        sv       = sv_list[cls_idx]
        col_off  = cls_idx * 3
        for i, feat in enumerate(top_feats):
            row      = i // 3
            col      = (i % 3) + col_off
            ax       = axes[row, col]
            feat_idx = list(X.columns).index(feat)
            ax.scatter(
                X[feat].values, sv[:, feat_idx],
                alpha=0.5, s=24, c="#4a90d9",
            )
            ax.set_xlabel(feat, fontsize=12)
            ax.set_ylabel("SHAP value", fontsize=12)
            ax.tick_params(axis="both", labelsize=10)
            ax.axhline(0, color="grey", linewidth=0.5, linestyle="--")
            if row == 0 and i % 3 == 0:
                ax.set_title(
                    PANEL_LABELS[cls_idx], fontsize=16, pad=10, loc="left",
                )

    # Hide unused subplots
    for r in range(n_rows):
        for c_off in range(2):
            for j in range(3):
                idx_flat = r * 3 + j
                if idx_flat >= n_feat:
                    axes[r, j + c_off * 3].set_visible(False)

    fig.tight_layout(pad=1.5)
    savefig_both(fig, f"{OUTDIR}/shap_dependence_{mkey}")
    plt.close()
    print(f"✓ shap_dependence_{mkey}.png / .pdf")

# ── 6e. Model Comparison Bar Chart (Macro F1) ───────────────────────────────
fig, ax = plt.subplots(figsize=(PANEL_W_IN, PANEL_W_IN * 0.4))
model_names = [MODELS[k][0] for k in MODELS]
macro_f1s   = [
    cv_sheets[k].loc[cv_sheets[k]["Fold"] == "Mean", "Macro F1"].values[0]
    for k in MODELS
]
macro_sds   = [
    cv_sheets[k].loc[cv_sheets[k]["Fold"] == "SD", "Macro F1"].values[0]
    for k in MODELS
]
bars = ax.barh(
    model_names[::-1], macro_f1s[::-1],
    xerr=macro_sds[::-1],
    color="#4a90d9", edgecolor="white", capsize=4,
)
ax.set_xlabel("Macro F1 (5-fold CV)", fontsize=16)
ax.tick_params(axis="both", labelsize=14)
ax.spines[["top", "right"]].set_visible(False)
ax.set_xlim(0, 1)
fig.tight_layout()
savefig_both(fig, f"{OUTDIR}/model_comparison_macrof1")
plt.close()
print("✓ model_comparison_macrof1.png / .pdf")

# ── 6f. SHAP Importance Comparison: Both Models Side-by-Side ────────────────
top10_per_model = {}
for mkey in MODELS:
    top10_per_model[mkey] = feat_imp_dfs[mkey].head(TOP_N_BAR)["Feature"].tolist()

# Union of top features, ordered by first appearance
all_top = []
for feats in top10_per_model.values():
    all_top.extend(feats)
unique_top = list(dict.fromkeys(all_top))

comp_data = {}
for mkey, (mname, _) in MODELS.items():
    imp_series = feat_imp_dfs[mkey].set_index("Feature")[
        "Mean |SHAP| (avg 2 classes)"
    ]
    comp_data[mname] = [imp_series.get(f, 0.0) for f in unique_top]

df_comp = pd.DataFrame(comp_data, index=unique_top)

fig, ax = plt.subplots(figsize=(PANEL_W_IN * 1.5, PANEL_W_IN * 0.9))
df_comp.iloc[::-1].plot.barh(ax=ax, edgecolor="white", width=0.8)
ax.set_xlabel("Mean |SHAP| (avg 2 classes)", fontsize=16)
ax.tick_params(axis="both", labelsize=13)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(fontsize=12, loc="lower right")
fig.tight_layout()
savefig_both(fig, f"{OUTDIR}/shap_importance_both_models")
plt.close()
print("✓ shap_importance_both_models.png / .pdf")

print("\n✓ All modelling outputs saved.")

✓ Found e-Ukraine font: 'e-Ukraine'
Target distribution (binary):
enjoyment_bin
Not enjoyable     42
Enjoyable        186
Name: count, dtype: int64

Feature matrix: 228 obs × 35 features
Target classes:  [ 42 186]  (Not enjoyable / Enjoyable)

─── CV: Random Forest ───
               precision    recall  f1-score   support

Not enjoyable       0.50      0.48      0.49        42
    Enjoyable       0.88      0.89      0.89       186

     accuracy                           0.82       228
    macro avg       0.69      0.68      0.69       228
 weighted avg       0.81      0.82      0.81       228

─── CV: CatBoost ───
               precision    recall  f1-score   support

Not enjoyable       0.45      0.45      0.45        42
    Enjoyable       0.88      0.88      0.88       186

     accuracy                           0.80       228
    macro avg       0.66      0.66      0.66       228
 weighted avg       0.80      0.80      0.80       228

─── SHAP: Random Forest ───
─── SHAP: CatBo

In [3]:
"""
Tables 2-3, Figure 7, Table S4.
1. Diverging bar: Enjoyable vs Not enjoyable feature prevalence
2. Association tests:
   A. Single-select NOMINAL → Chi-square + Cramér's V
   B. Single-select ORDINAL → Cochran–Armitage trend + Chi-square
   C. Multi-select → per-option Fisher/Chi-square + BH FDR correction
"""

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.patches import Patch
from scipy.stats import chi2_contingency, fisher_exact, norm
from statsmodels.stats.multitest import multipletests
import warnings
warnings.filterwarnings("ignore")

# ── Style + e-Ukraine Font ───────────────────────────────────────────────────
_eu_fonts = [f for f in fm.findSystemFonts()
             if "e-Ukraine" in f or "eUkraine" in f or "e_Ukraine" in f]
if _eu_fonts:
    for fp in _eu_fonts:
        fm.fontManager.addfont(fp)
    _eu_name = fm.FontProperties(fname=_eu_fonts[0]).get_name()
else:
    _eu_name = "e-Ukraine"

plt.rcParams.update({
    "font.family": _eu_name,
    "font.size": 11,
    "axes.titlesize": 9,
    "axes.labelsize": 9,
    "figure.dpi": 300,
})

OUTDIR = "C:/Users/karasoo1/Ukrainability"
W_CM   = 18
W_IN   = W_CM / 2.54
DPI_PNG = 900
PNTD   = "Prefer not to disclose"

CB_COLORS = {"Not enjoyable": "#D55E00", "Enjoyable": "#009E73"}


def _save(fig, stem):
    for ext in ("png", "pdf"):
        fig.savefig(f"{OUTDIR}/{stem}.{ext}", bbox_inches="tight",
                    dpi=DPI_PNG if ext == "png" else None)
    plt.close(fig)
    print(f"✓ {stem}.png / .pdf")


# ═══════════════════════════════════════════════════════════════════════════
# 0. Load & Prepare
# ═══════════════════════════════════════════════════════════════════════════
df = pd.read_excel(f"{OUTDIR}/english_data_for_sharing.xlsx", sheet_name="english_data")

emap = {
    "Not enjoyable at all": "Not enjoyable",
    "Not enjoyable":        "Not enjoyable",
    "Neutral":              "Not enjoyable",
    "Enjoyable":            "Enjoyable",
    "Very enjoyable":       "Enjoyable",
}
df["enjoyment_bin"] = df["enjoyment"].map(emap)
target_order = ["Not enjoyable", "Enjoyable"]
df["duration_visit"] = df["duration_visit"].replace("1–2 hours", "1-2 hours")


# ═══════════════════════════════════════════════════════════════════════════
# 1. DIVERGING BAR: ENJOYABLE vs NOT ENJOYABLE (Vertical)
# ═══════════════════════════════════════════════════════════════════════════

pos_mask    = df["enjoyment_bin"] == "Enjoyable"
nonpos_mask = df["enjoyment_bin"] == "Not enjoyable"

records = []

for col, prefix in [("purpose_visit", "Act"), ("accessibility", "Acc"),
                    ("visitor_type", "Vis")]:
    all_opts = set()
    for v in df[col].dropna():
        for item in str(v).split(";"):
            all_opts.add(item.strip())
    for opt in sorted(all_opts):
        def has_opt(x, _opt=opt):
            return _opt in str(x).split(";") if pd.notna(x) else False
        pct_pos    = df.loc[pos_mask, col].apply(has_opt).mean() * 100
        pct_nonpos = df.loc[nonpos_mask, col].apply(has_opt).mean() * 100
        records.append({"feature": f"{prefix}: {opt}",
                        "pct_enjoyable": pct_pos,
                        "pct_not_enjoyable": pct_nonpos,
                        "diff_pp": pct_pos - pct_nonpos})

for col, prefix in [("age", "Age"), ("gender", "Gen"), ("regularity", "Reg"),
                    ("duration_visit", "Dur"), ("noticed_changes", "Chg"),
                    ("occupation", "Occ")]:
    for cat in df[col].dropna().unique():
        if cat == PNTD:
            continue
        pct_pos    = (df.loc[pos_mask, col] == cat).mean() * 100
        pct_nonpos = (df.loc[nonpos_mask, col] == cat).mean() * 100
        records.append({"feature": f"{prefix}: {cat}",
                        "pct_enjoyable": pct_pos,
                        "pct_not_enjoyable": pct_nonpos,
                        "diff_pp": pct_pos - pct_nonpos})

div_df = pd.DataFrame(records)
div_df = div_df[div_df["diff_pp"].abs() >= 3].copy()
div_df = div_df.sort_values("diff_pp", ascending=False)

n_bars = len(div_df)
fig, ax = plt.subplots(figsize=(W_IN, W_IN * 0.75))

colors = [CB_COLORS["Enjoyable"] if v >= 0 else CB_COLORS["Not enjoyable"]
          for v in div_df["diff_pp"]]

ax.bar(range(n_bars), div_df["diff_pp"].values, color=colors,
       edgecolor="white", width=0.7)

ax.set_xticks(range(n_bars))
ax.set_xticklabels(div_df["feature"].values, rotation=45, ha="right", fontsize=8)
ax.axhline(0, color="black", linewidth=0.8)

ax.set_ylabel("Difference (pp): Enjoyable minus Not enjoyable", fontsize=10)
ax.spines[["top", "right"]].set_visible(False)
ax.tick_params(axis="y", labelsize=9)

max_val = div_df["diff_pp"].abs().max()
ax.set_ylim(-max_val * 1.3, max_val * 1.3)

for i, v in enumerate(div_df["diff_pp"].values):
    va     = "bottom" if v >= 0 else "top"
    offset = max_val * 0.03 if v >= 0 else -max_val * 0.03
    ax.text(i, v + offset, f"{v:+.1f}", va=va, ha="center",
            fontsize=8, rotation=90)

legend_handles = [
    Patch(facecolor=CB_COLORS["Enjoyable"],
          label="More common among Enjoyable"),
    Patch(facecolor=CB_COLORS["Not enjoyable"],
          label="More common among Not enjoyable"),
]
ax.legend(handles=legend_handles, bbox_to_anchor=(0.5, 1.12),
          loc="upper center", ncol=2, fontsize=9, framealpha=0.9,
          borderpad=0.8, edgecolor="lightgrey")

fig.tight_layout()
_save(fig, "diverging_enjoyable_vs_not_enjoyable")

# Full table (unfiltered)
div_full = pd.DataFrame(records).sort_values("diff_pp", ascending=False)
div_full.to_csv(f"{OUTDIR}/enjoyable_vs_not_enjoyable.csv", index=False)
print("✓ enjoyable_vs_not_enjoyable.csv")


# ═══════════════════════════════════════════════════════════════════════════
# 2. ASSOCIATION TESTS WITH ENJOYMENT (BINARY)
# ═══════════════════════════════════════════════════════════════════════════
#
# Three families of tests, matched to variable type:
#
#   A. Single-select NOMINAL (unordered categories):
#      → Chi-square test of independence + Cramér's V
#      Variables: gender, occupation, noticed_changes
#
#   B. Single-select ORDINAL (ordered categories):
#      → Cochran–Armitage trend test (exact 1-df trend) + Cramér's V
#      → Also report standard chi-square for comparison
#      Variables: age, income, duration_visit, regularity
#
#   C. Multi-select (semicolon-separated; responses not mutually exclusive):
#      → Per-option 2×2 Fisher's exact test (or chi-square w/ Yates)
#      → Benjamini–Hochberg FDR correction across options within each variable
#      Variables: purpose_visit, visitor_type, accessibility, wishlist
#

# ── Helper: Cochran-Armitage trend test ──────────────────────────────────
def cochran_armitage_trend(ct_ordered, scores=None):
    """
    Cochran–Armitage test for trend in a 2×K table.
    ct_ordered: pd.DataFrame with 2 rows (binary outcome) and K ordered columns.
    scores: integer scores for each column (default: 0, 1, 2, ..., K-1).
    Returns: Z-statistic, two-sided p-value.
    """
    tab = ct_ordered.values  # shape (2, K)
    if tab.shape[0] != 2:
        raise ValueError("Need exactly 2 rows (binary outcome)")
    K = tab.shape[1]
    if scores is None:
        scores = np.arange(K, dtype=float)
    else:
        scores = np.array(scores, dtype=float)

    n_j  = tab.sum(axis=0)
    N    = tab.sum()
    p1_j = tab[0]
    p_hat = tab[0].sum() / N

    T  = np.sum(scores * p1_j)
    T0 = p_hat * np.sum(scores * n_j)
    V  = p_hat * (1 - p_hat) * (
        np.sum(scores**2 * n_j) - (np.sum(scores * n_j))**2 / N
    )

    if V <= 0:
        return 0.0, 1.0
    Z = (T - T0) / np.sqrt(V)
    p_value = 2 * norm.sf(np.abs(Z))
    return Z, p_value


# ── A. Single-select NOMINAL variables ───────────────────────────────────
print("\n" + "=" * 75)
print("A. SINGLE-SELECT NOMINAL VARIABLES: Chi-square test of independence")
print("=" * 75)

nominal_vars = ["gender", "occupation", "noticed_changes"]
nominal_results = []

for col in nominal_vars:
    ct = pd.crosstab(df[col], df["enjoyment_bin"])
    ct = ct.drop(PNTD, errors="ignore")
    ct = ct[ct.sum(axis=1) >= 5]
    if ct.shape[0] < 2:
        print(f"  {col:25s}  skipped (fewer than 2 categories with n≥5)")
        continue
    chi2, p, dof, _ = chi2_contingency(ct)
    n = ct.values.sum()
    V = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    sig   = "Yes" if p < 0.05 else "No"
    stars = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    nominal_results.append({
        "Variable": col, "Test": "Chi-square", "Statistic": round(chi2, 2),
        "df": dof, "p-value": round(p, 4), "Cramér's V": round(V, 3),
        "Significant": sig,
    })
    print(f"  {col:25s}  χ²={chi2:7.2f}  df={dof}  p={p:.4f}  V={V:.3f}  {stars}")


# ── B. Single-select ORDINAL variables ───────────────────────────────────
print(f"\n{'=' * 75}")
print("B. SINGLE-SELECT ORDINAL VARIABLES: Cochran-Armitage trend test")
print("   (Chi-square also reported for comparison)")
print("=" * 75)

ordinal_orders = {
    "age":            ["18-25", "26-40", "41-60", "60+"],
    "income":         ["0-5000 UAH", "5001-10000 UAH",
                       "10001-20000 UAH", "20000+ UAH"],
    "duration_visit": ["Passing by", "1-2 hours", "Most of the day",
                       "Longer trip", "Few days"],
    "regularity":     ["One-time visit", "Before war only", "Occasionally",
                       "Regularly", "Frequently", "Very frequently"],
}

ordinal_results = []

for col, order in ordinal_orders.items():
    ct = pd.crosstab(df[col], df["enjoyment_bin"])
    ct = ct.drop(PNTD, errors="ignore")
    present = [o for o in order if o in ct.index and ct.loc[o].sum() >= 5]
    if len(present) < 2:
        print(f"  {col:25s}  skipped (fewer than 2 ordered categories with n≥5)")
        continue
    ct_ord = (ct.loc[present]
              .reindex(columns=["Not enjoyable", "Enjoyable"])
              .fillna(0).astype(int))

    Z, p_trend = cochran_armitage_trend(
        ct_ord.T, scores=list(range(len(present))),
    )
    n = ct_ord.values.sum()

    chi2, p_chi2, dof, _ = chi2_contingency(ct_ord)
    V = np.sqrt(chi2 / (n * (min(ct_ord.shape) - 1)))

    stars_t = "***" if p_trend < 0.001 else "**" if p_trend < 0.01 else "*" if p_trend < 0.05 else "ns"
    stars_c = "***" if p_chi2 < 0.001 else "**" if p_chi2 < 0.01 else "*" if p_chi2 < 0.05 else "ns"

    ordinal_results.append({
        "Variable": col,
        "Test": "Cochran-Armitage trend",
        "Z": round(Z, 3),
        "p-value (trend)": round(p_trend, 4),
        "Chi2": round(chi2, 2),
        "df": dof,
        "p-value (chi2)": round(p_chi2, 4),
        "Cramér's V": round(V, 3),
        "N categories": len(present),
        "Trend significant": "Yes" if p_trend < 0.05 else "No",
        "Chi2 significant": "Yes" if p_chi2 < 0.05 else "No",
    })
    print(f"  {col:25s}  Trend: Z={Z:+.3f} p={p_trend:.4f} {stars_t}"
          f"  |  χ²={chi2:7.2f} df={dof} p={p_chi2:.4f} V={V:.3f} {stars_c}")


# ── C. Multi-select variables: per-option 2×2 tests ─────────────────────
print(f"\n{'=' * 75}")
print("C. MULTI-SELECT VARIABLES: Per-option Fisher exact / Chi-square tests")
print("   (Benjamini-Hochberg FDR correction within each variable)")
print("=" * 75)

multi_vars = ["purpose_visit", "visitor_type", "accessibility", "wishlist"]
multi_results = []

for col in multi_vars:
    all_opts = set()
    for v in df[col].dropna():
        for item in str(v).split(";"):
            s = item.strip()
            if s and s != PNTD:
                all_opts.add(s)
    all_opts = sorted(all_opts)

    option_tests = []
    for opt in all_opts:
        has_opt = df[col].apply(
            lambda x, _o=opt: _o in str(x).split(";") if pd.notna(x) else False
        )
        ct = pd.crosstab(has_opt, df["enjoyment_bin"])
        ct = (ct.reindex(index=[True, False],
                         columns=["Not enjoyable", "Enjoyable"])
              .fillna(0).astype(int))

        n_selected = int(has_opt.sum())
        if n_selected < 5:
            continue

        expected_min = (
            ct.sum(axis=1).values.reshape(-1, 1)
            * ct.sum(axis=0).values.reshape(1, -1)
            / ct.values.sum()
        ).min()

        if expected_min < 5:
            odds_ratio, p_val = fisher_exact(ct.values)
            test_used = "Fisher"
            stat = odds_ratio
        else:
            chi2, p_val, dof, _ = chi2_contingency(ct, correction=True)
            test_used = "Chi2-Yates"
            stat = chi2

        pct_enjoy_yes = (ct.loc[True, "Enjoyable"] / ct.loc[True].sum() * 100
                         if ct.loc[True].sum() > 0 else 0)
        pct_enjoy_no  = (ct.loc[False, "Enjoyable"] / ct.loc[False].sum() * 100
                         if ct.loc[False].sum() > 0 else 0)

        option_tests.append({
            "Variable": col,
            "Option": opt,
            "n": n_selected,
            "Test": test_used,
            "Statistic": round(stat, 3),
            "p_raw": p_val,
            "% Enjoyable (selected)": round(pct_enjoy_yes, 1),
            "% Enjoyable (not selected)": round(pct_enjoy_no, 1),
            "Δ (pp)": round(pct_enjoy_yes - pct_enjoy_no, 1),
        })

    if not option_tests:
        continue

    p_raw = [t["p_raw"] for t in option_tests]
    reject, p_adj, _, _ = multipletests(p_raw, method="fdr_bh", alpha=0.05)

    print(f"\n  {col} ({len(option_tests)} options tested):")
    for i, t in enumerate(option_tests):
        t["p_adjusted (BH)"] = round(p_adj[i], 4)
        t["p_raw"] = round(t["p_raw"], 4)
        t["Significant (FDR<0.05)"] = "Yes" if reject[i] else "No"
        multi_results.append(t)
        stars = ("***" if p_adj[i] < 0.001 else "**" if p_adj[i] < 0.01
                 else "*" if p_adj[i] < 0.05 else "ns")
        print(f"    {t['Option']:30s}  n={t['n']:3d}  "
              f"Enjoy: {t['% Enjoyable (selected)']:5.1f}% "
              f"vs {t['% Enjoyable (not selected)']:5.1f}%  "
              f"Δ={t['Δ (pp)']:+5.1f}pp  "
              f"p_raw={t['p_raw']:.4f}  p_adj={p_adj[i]:.4f}  {stars}")


# ── Combine and save all results ─────────────────────────────────────────
print(f"\n{'=' * 75}")
print("SUMMARY TABLES")
print("=" * 75)

single_results = []
for r in nominal_results:
    single_results.append({
        "Variable": r["Variable"],
        "Type": "Nominal",
        "Test": "Chi-square",
        "Statistic": r["Statistic"],
        "df": r["df"],
        "p-value": r["p-value"],
        "Trend p": "—",
        "Cramér's V": r["Cramér's V"],
        "Significant (α=0.05)": r["Significant"],
    })
for r in ordinal_results:
    single_results.append({
        "Variable": r["Variable"],
        "Type": "Ordinal",
        "Test": "Chi-square + Trend",
        "Statistic": r["Chi2"],
        "df": r["df"],
        "p-value": r["p-value (chi2)"],
        "Trend p": r["p-value (trend)"],
        "Cramér's V": r["Cramér's V"],
        "Significant (α=0.05)": r["Chi2 significant"],
    })

single_df = pd.DataFrame(single_results)
single_df.to_csv(f"{OUTDIR}/association_tests_single_select.csv", index=False)
print(f"\n  Table 1 — Single-select variables ({len(single_df)} rows):")
print(single_df.to_string(index=False))

multi_df = pd.DataFrame(multi_results)
multi_df.to_csv(f"{OUTDIR}/association_tests_multi_select.csv", index=False)
print(f"\n  Table 2 — Multi-select variables ({len(multi_df)} rows):")
print(multi_df[["Variable", "Option", "n", "% Enjoyable (selected)",
                "% Enjoyable (not selected)", "Δ (pp)", "p_raw",
                "p_adjusted (BH)", "Significant (FDR<0.05)"]].to_string(index=False))

print(f"\n✓ association_tests_single_select.csv")
print(f"✓ association_tests_multi_select.csv")
print("\n✓ All narrative analysis outputs saved.")

✓ diverging_enjoyable_vs_not_enjoyable.png / .pdf
✓ enjoyable_vs_not_enjoyable.csv

A. SINGLE-SELECT NOMINAL VARIABLES: Chi-square test of independence
  gender                     χ²=   2.94  df=1  p=0.0866  V=0.115  ns
  occupation                 χ²=   7.64  df=4  p=0.1057  V=0.190  ns
  noticed_changes            χ²=  14.36  df=2  p=0.0008  V=0.258  ***

B. SINGLE-SELECT ORDINAL VARIABLES: Cochran-Armitage trend test
   (Chi-square also reported for comparison)
  age                        Trend: Z=-2.142 p=0.0322 *  |  χ²=   4.82 df=3 p=0.1855 V=0.147 ns
  income                     Trend: Z=-1.256 p=0.2092 ns  |  χ²=   7.95 df=3 p=0.0470 V=0.224 *
  duration_visit             Trend: Z=+0.190 p=0.8495 ns  |  χ²=  11.85 df=3 p=0.0079 V=0.233 **
  regularity                 Trend: Z=+1.667 p=0.0956 ns  |  χ²=   7.60 df=4 p=0.1074 V=0.185 ns

C. MULTI-SELECT VARIABLES: Per-option Fisher exact / Chi-square tests
   (Benjamini-Hochberg FDR correction within each variable)

  purpose_vi